# 팀 실습 — 공개 정리본
교육 과정의 제공 골격을 바탕으로 수행한 실습입니다. 개인/팀 범위를 폴더로 구분했습니다.
공개 정리 시 환경 연결 셀·데이터 미리보기·내장 교안 이미지를 제외했습니다. 학습 설정과 모델 로직은 원본을 유지합니다.
개인 노트북의 평가 텍스트는 당시 저장 결과이며 나머지 실행 출력은 정리했습니다.


In [ ]:
import os


# **Step3. 속성 기반 감정 분류**

**미션 : 리뷰 안에 담긴 11가지 속성 감성 분석하기(Aspect-Based Sentiment Analysis)**
1) 목표 : 속성 기반 감성 분석을 통해, 리뷰 내용을 속성에 따라 긍, 부정
으로 분류하는 모델 생성
2) 절차① : “[ASPECT] 속성 [SEP] 문장”
    * 데이터 전처리 : 데이터셋 구성, 토크나이즈
    * 스페셜 토큰 추가 + 모델의 임베딩 층 resizing
    * 파인튜닝 및 모델 평가
3) 절차② : “Aspect : 문장”
    * 데이터 전처리 : 데이터셋 구성, 토크나이즈
    * 파인튜닝 및 모델 평가
4) 성능 비교 : “Aspect : 문장”
    * 모델 별 : klue/bert-base, klue/RoBERTa-base
    * 전처리 별 : Case①, Case②
    * Macro Recall, Macro Precision, Macro F1-Score 비교

## 1.환경준비

### (1) 라이브러리 설치 및 로딩

In [ ]:
!apt -qq -y install fonts-nanum > /dev/null
!rm -rf ~/.cache/matplotlib

import matplotlib as mpl
import matplotlib.font_manager as fm
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
mpl.rcParams['font.family'] = 'NanumGothic'
mpl.rcParams['axes.unicode_minus'] = False

In [ ]:
# 한글폰트 설치
import matplotlib as mpl
import matplotlib.font_manager as fm

%config InlineBackend.figure_format = 'retina'
!apt -qq -y install fonts-nanum

font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fontprop = fm.FontProperties(fname=font_path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import torch

from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder

import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

### (2) 데이터 로딩
* data.csv 파일 로딩

In [ ]:
# 공개본: 환경 연결 셀은 생략. 데이터 경로는 DATA_PATH로 지정합니다.


* 전제 데이터 중 10,000건만 샘플링
    * 학습 속도 고려

In [ ]:
data_sample = data.sample(10000)

### (3) GPU 설정
* GPU 사용을 위한 설정

In [ ]:
# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2.방식① : [ASPECT] 속성 [SEP] 문장

* 두가지 모델에 대해서 튜닝을 시도해봅시다.
    * model_name1 = "klue/bert-base"
    * model_name2 = "klue/roberta-base"

### (1) 데이터 준비
* 데이터셋 분할 및 텐서 데이터셋 변환
* 토크나이징 함수 안에서,
    * "[ASPECT] 속성 [SEP] 문장" 형태로 수정
    * 토크나이징

#### 1) 데이터셋 분할 및 텐서 데이터셋 변환

In [ ]:
aspects = ['보습', '기능', '밀착', '가격', '향', '편의성', '기타', '외형', '포장', '품질', '피부']

In [ ]:
# train, val 데이터 분할
train, val = train_test_split(data_sample, test_size=0.3, random_state=42)

In [ ]:
# df로 부터 텐서 데이터셋 만들기
train_ts = Dataset.from_pandas(train)
val_ts = Dataset.from_pandas(val)

In [ ]:
train_ts[:3]

#### 2) 토크나이징
* "[ASPECT] 속성 [SEP] 문장" 형태로 수정
* 다중 레이블 분류 문제에서 label은 float 타입이어야 함

* 토크나이저 준비 : 두 모델에 대한 토크나이저 로딩

In [ ]:
model_name1 = 'klue/bert-base'
model_name2 = 'klue/roberta-base'
c1_tokenizer_bert = AutoTokenizer.from_pretrained(model_name1)
c1_tokenizer_roberta = AutoTokenizer.from_pretrained(model_name2)

* 사용자 정의 스페셜 토큰 등록하기

In [ ]:
# 사용자 정의 스페셜 토큰 목록
special_tokens_dict = {'additional_special_tokens':['[ASPECT]']}

# 토크나이저에 추가
c1_num_added_toks_bert = c1_tokenizer_bert.add_special_tokens(special_tokens_dict)
print(f"Added {c1_num_added_toks_bert} special tokens_bert: {special_tokens_dict['additional_special_tokens']}")

c1_num_added_toks_roberta = c1_tokenizer_roberta.add_special_tokens(special_tokens_dict)
print(f"Added {c1_num_added_toks_roberta} special tokens_roberta: {special_tokens_dict['additional_special_tokens']}")

* 토크나이징을 포함한 함수 생성 및 적용

In [ ]:
# 모델1 전처리 함수
def preprocess_function_bert(data):
    texts = data['text']
    aspects = data['aspect']

    combined_inputs = [f"[ASPECT] {asp} [SEP] {txt}" for asp, txt in zip(aspects, texts)]
    tokenized = c1_tokenizer_bert(combined_inputs, truncation=True, padding='max_length', max_length=128)

    tokenized['labels'] = data['label']

    return tokenized

# 모델1 전처리 적용
c1_train_ts_bert = train_ts.map(preprocess_function_bert, batched=True)
c1_val_ts_bert = val_ts.map(preprocess_function_bert, batched=True)


# 모델2 전처리 함수
def preprocess_function_roberta(data):
    texts = data['text']
    aspects = data['aspect']

    combined_inputs = [f"[ASPECT] {asp} [SEP] {txt}" for asp, txt in zip(aspects, texts)]
    tokenized = c1_tokenizer_roberta(combined_inputs, truncation=True, padding='max_length', max_length=128)

    tokenized['labels'] = data['label']

    return tokenized


# 모델2 전처리 적용
c1_train_ts_roberta = train_ts.map(preprocess_function_roberta, batched=True)
c1_val_ts_roberta = val_ts.map(preprocess_function_roberta, batched=True)



In [ ]:
# 🚨 주의: 현재 코드 블록 내에서 c1_tokenizer_bert와 c1_tokenizer_roberta가 정의되어 있어야 합니다.

print("--- 🤖 KLUE-BERT 전처리 결과 (첫 번째 샘플) ---")
# c1_train_ts_bert (Map 결과)의 첫 번째 요소에서 'input_ids'를 가져와 디코딩
print("Input Text (Decoded):")
print(c1_tokenizer_bert.decode(c1_train_ts_bert[0]['input_ids']))
print("\nLabels:")
print(c1_train_ts_bert[0]['labels'])

print("\n" + "="*50 + "\n")

print("--- 🚀 KLUE-RoBERTa 전처리 결과 (첫 번째 샘플) ---")
# c1_train_ts_roberta (Map 결과)의 첫 번째 요소에서 'input_ids'를 가져와 디코딩
print("Input Text (Decoded):")
print(c1_tokenizer_roberta.decode(c1_train_ts_roberta[0]['input_ids']))
print("\nLabels:")
print(c1_train_ts_roberta[0]['labels'])

### (2) Fine-Tuning

#### 1) 사전학습 모델 준비
* 모델 로드한 후
* 반드시 `모델.resize_token_embeddings(len(tokenizer))`

In [ ]:
# 모델1
c1_model_bert = AutoModelForSequenceClassification.from_pretrained(model_name1, num_labels=2)
c1_model_bert.resize_token_embeddings(len(c1_tokenizer_bert))

In [ ]:
# 모델2
c1_model_roberta = AutoModelForSequenceClassification.from_pretrained(model_name2, num_labels=2)
c1_model_roberta.resize_token_embeddings(len(c1_tokenizer_roberta))

#### 2) 학습 설정

* TrainingArguments 설정
    * 두 모델에 공통으로 하나만 설정해도 됨

In [ ]:
training_args = TrainingArguments(
    output_dir = './results_mini2_c1',
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 5e-5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    weight_decay = 0.02,
    load_best_model_at_end = True,
    logging_dir ='./logs_mini2_c1',
    logging_steps = 10,
    report_to="tensorboard"
)

* Trainer 설정
    * 두 모델 각각 설정

In [ ]:
# Trainer1 설정
trainer1 = Trainer(
    model=c1_model_bert,
    args=training_args,
    train_dataset = c1_train_ts_bert,
    eval_dataset = c1_val_ts_bert,
    tokenizer = c1_tokenizer_bert
)


# Trainer2 설정
trainer2 = Trainer(
    model=c1_model_roberta,
    args=training_args,
    train_dataset = c1_train_ts_roberta,
    eval_dataset = c1_train_ts_roberta,
    tokenizer = c1_tokenizer_roberta
)


#### 3) 학습

In [ ]:
trainer1.train()

In [ ]:
trainer2.train()

#### 4) 모델 검증평가

* 예측

In [ ]:
c1_raw_pred_bert = trainer1.predict(c1_val_ts_bert)
c1_raw_pred_roberta = trainer2.predict(c1_val_ts_roberta)

In [ ]:
c1_pred_bert = c1_raw_pred_bert.predictions.argmax(axis=1)
c1_pred_roberta = c1_raw_pred_roberta.predictions.argmax(axis=1)

In [ ]:
c1_true_bert = c1_raw_pred_bert.label_ids
c1_true_roberta = c1_raw_pred_roberta.label_ids

* 평가 : model1

In [ ]:
try:
    aspect_list = val_ts['aspect']
except NameError:
    print("오류: 'val_ts', 'true1', 'pred1' 변수를 찾을 수 없습니다.")
    print("이전 단계의 '예측' 셀을 먼저 실행했는지 확인해주세요.")
    raise

df_eval1 = pd.DataFrame({
    'aspect': aspect_list,
    'y_true': c1_true_bert,
    'y_pred': c1_pred_bert
})

In [ ]:
metrics_list1 = []
unique_aspects1 = sorted(df_eval1['aspect'].unique())
target_names = ['부정 (0)', '긍정 (1)']

print(f"\n[모델 1] 총 {len(unique_aspects1)}개의 속성에 대해 개별 Classification Report를 출력합니다.")
print("=" * 60)

for aspect in unique_aspects1:
    aspect_data = df_eval1[df_eval1['aspect'] == aspect]

    if len(aspect_data) == 0:
        print(f"\n--- '{aspect}' 속성: 데이터 없음 (스킵) ---")
        continue

    y_true_aspect = aspect_data['y_true']
    y_pred_aspect = aspect_data['y_pred']

    print(f"\n--- [모델 1] '{aspect}' 속성 Classification Report ---")
    print(classification_report(
        y_true_aspect,
        y_pred_aspect,
        target_names=target_names,
        zero_division=0
    ))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_aspect,
        y_pred_aspect,
        average='macro',
        zero_division=0
    )

    metrics_list1.append({
        'aspect': aspect,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    })

print("=" * 60)
print("모든 속성별 리포트 출력이 완료되었습니다.")

In [ ]:
df_metrics1 = pd.DataFrame(metrics_list1)
df_metrics1_long = df_metrics1.melt(
    id_vars='aspect',
    value_vars=['precision', 'recall', 'f1_score'],
    var_name='metric',
    value_name='score'
)

In [ ]:
print("\n[모델 1] 시각화 그래프를 생성합니다...")
plt.figure(figsize=(18, 8)) # 그래프 크기 설정

# 그룹 막대 그래프 생성
sns.barplot(
    data=df_metrics1_long,
    x='aspect',
    y='score',
    hue='metric'
)

plt.title('[모델 1] 속성(Aspect)별 Macro 점수 비교 (F1, Precision, Recall)', fontsize=16)
plt.xlabel('속성 (Aspect)', fontsize=12)
plt.ylabel('점수 (Score)', fontsize=12)
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Metric', loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

* 평가 : model2

In [ ]:
try:
    aspect_list = val_ts['aspect']
except NameError:
    print("오류: 'val_ts', 'true2', 'pred2' 변수를 찾을 수 없습니다.")
    raise

df_eval2 = pd.DataFrame({
    'aspect': aspect_list,
    'y_true': c1_true_roberta,
    'y_pred': c1_pred_roberta
})

In [ ]:
metrics_list2 = []
unique_aspects2 = sorted(df_eval2['aspect'].unique())
target_names = ['부정 (0)', '긍정 (1)']

print(f"\n[모델 2] 총 {len(unique_aspects2)}개의 속성에 대해 개별 Classification Report를 출력합니다.")
print("=" * 60)

for aspect in unique_aspects2:
    aspect_data = df_eval2[df_eval2['aspect'] == aspect]

    if len(aspect_data) == 0:
        print(f"\n--- '{aspect}' 속성: 데이터 없음 (스킵) ---")
        continue

    y_true_aspect = aspect_data['y_true']
    y_pred_aspect = aspect_data['y_pred']

    print(f"\n--- [모델 2] '{aspect}' 속성 Classification Report ---")
    print(classification_report(
        y_true_aspect,
        y_pred_aspect,
        target_names=target_names,
        zero_division=0
    ))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_aspect,
        y_pred_aspect,
        average='macro',
        zero_division=0
    )

    metrics_list2.append({
        'aspect': aspect,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    })

print("=" * 60)
print("모든 속성별 리포트 출력이 완료되었습니다.")

In [ ]:
df_metrics2 = pd.DataFrame(metrics_list2)
df_metrics2_long = df_metrics2.melt(
    id_vars='aspect',
    value_vars=['precision', 'recall', 'f1_score'],
    var_name='metric',
    value_name='score'
)

In [ ]:
print("\n[모델 2] 시각화 그래프를 생성합니다...")
plt.figure(figsize=(18, 8))

sns.barplot(
    data=df_metrics2_long,
    x='aspect',
    y='score',
    hue='metric'
)

plt.title('[모델 2] 속성(Aspect)별 Macro 점수 비교 (F1, Precision, Recall)', fontsize=16)
plt.xlabel('속성 (Aspect)', fontsize=12)
plt.ylabel('점수 (Score)', fontsize=12)
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Metric', loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 3.방식② : 속성 : 문장

* 두가지 모델에 대해서 튜닝을 시도해봅시다.
    * model_name1 = "klue/bert-base"
    * model_name2 = "klue/roberta-base"

### (1) 데이터 준비
* 데이터셋 분할 및 텐서 데이터셋 변환
* 토크나이징 함수 안에서,
    * "ASPECT : 문장" 형태로 수정
    * 토크나이징

#### 1) 데이터셋 분할 및 텐서 데이터셋 변환

In [ ]:
aspects = ['보습', '기능', '밀착', '가격', '향', '편의성', '기타', '외형', '포장', '품질', '피부']

In [ ]:
# train, val 데이터 분할
train, val = train_test_split(data_sample, test_size=0.3, random_state=42)

In [ ]:
# df로 부터 텐서 데이터셋 만들기
train_ts = Dataset.from_pandas(train)
val_ts = Dataset.from_pandas(val)

In [ ]:
train_ts[:3]

#### 2) 토크나이징
* "ASPECT : 문장" 형태로 수정
* 다중 레이블 분류 문제에서 label은 float 타입이어야 함

* 토크나이저 준비 : 두 모델에 대한 토크나이저 로딩

In [ ]:
model_name1 = 'klue/bert-base'
model_name2 = 'klue/roberta-base'
c2_tokenizer_bert = AutoTokenizer.from_pretrained(model_name1)
c2_tokenizer_roberta = AutoTokenizer.from_pretrained(model_name2)

In [ ]:
# 사용자 정의 스페셜 토큰 목록
special_tokens_dict = {'additional_special_tokens':['[ASPECT]']}

# 토크나이저에 추가
c2_num_added_toks_bert = c1_tokenizer_bert.add_special_tokens(special_tokens_dict)
print(f"Added {c1_num_added_toks_bert} special tokens_bert: {special_tokens_dict['additional_special_tokens']}")

c2_num_added_toks_roberta = c1_tokenizer_roberta.add_special_tokens(special_tokens_dict)
print(f"Added {c1_num_added_toks_roberta} special tokens_roberta: {special_tokens_dict['additional_special_tokens']}")

* 토크나이징을 포함한 함수 생성 및 적용

In [ ]:
# 모델1 전처리 함수
def preprocess_function_bert(data):
    texts = data['text']
    aspects = data['aspect']

    combined_inputs = [f"[ASPECT] {asp} [SEP] {txt}" for asp, txt in zip(aspects, texts)]
    tokenized = c1_tokenizer_bert(combined_inputs, truncation=True, padding='max_length', max_length=128)

    tokenized['labels'] = data['label']

    return tokenized

# 모델1 전처리 적용
c2_train_ts_bert = train_ts.map(preprocess_function_bert, batched=True)
c2_val_ts_bert = val_ts.map(preprocess_function_bert, batched=True)


# 모델2 전처리 함수
def preprocess_function_roberta(data):
    texts = data['text']
    aspects = data['aspect']

    combined_inputs = [f"[ASPECT] {asp} [SEP] {txt}" for asp, txt in zip(aspects, texts)]
    tokenized = c1_tokenizer_roberta(combined_inputs, truncation=True, padding='max_length', max_length=128)

    tokenized['labels'] = data['label']

    return tokenized


# 모델2 전처리 적용
c2_train_ts_roberta = train_ts.map(preprocess_function_roberta, batched=True)
c2_val_ts_roberta = val_ts.map(preprocess_function_roberta, batched=True)




In [ ]:
# 🚨 주의: 현재 코드 블록 내에서 c1_tokenizer_bert와 c1_tokenizer_roberta가 정의되어 있어야 합니다.

print("--- 🤖 KLUE-BERT 전처리 결과 (첫 번째 샘플) ---")
# c1_train_ts_bert (Map 결과)의 첫 번째 요소에서 'input_ids'를 가져와 디코딩
print("Input Text (Decoded):")
print(c2_tokenizer_bert.decode(c2_train_ts_bert[0]['input_ids']))
print("\nLabels:")
print(c2_train_ts_bert[0]['labels'])

print("\n" + "="*50 + "\n")

print("--- 🚀 KLUE-RoBERTa 전처리 결과 (첫 번째 샘플) ---")
# c1_train_ts_roberta (Map 결과)의 첫 번째 요소에서 'input_ids'를 가져와 디코딩
print("Input Text (Decoded):")
print(c2_tokenizer_roberta.decode(c2_train_ts_roberta[0]['input_ids']))
print("\nLabels:")
print(c2_train_ts_roberta[0]['labels'])

### (2) Fine-Tuning

#### 1) 사전학습 모델 준비

In [ ]:
# 모델1
c2_model_bert = AutoModelForSequenceClassification.from_pretrained(model_name1, num_labels=2)
c2_model_bert.resize_token_embeddings(len(c1_tokenizer_bert))

In [ ]:
# 모델2
c2_model_roberta = AutoModelForSequenceClassification.from_pretrained(model_name2, num_labels=2)
c2_model_roberta.resize_token_embeddings(len(c1_tokenizer_roberta))

#### 2) 학습 설정

* TrainingArguments 설정
    * 두 모델에 공통으로 하나만 설정해도 됨

In [ ]:
training_args = TrainingArguments(
    output_dir = './results_mini2_c1',
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 5e-5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    weight_decay = 0.02,
    load_best_model_at_end = True,
    logging_dir ='./logs_mini2_c1',
    logging_steps = 10,
    report_to="tensorboard"
)

* Trainer 설정
    * 두 모델 각각 설정

In [ ]:
# Trainer1 설정
c2_trainer1 = Trainer(
    model=c2_model_bert,
    args=training_args,
    train_dataset = c2_train_ts_bert,
    eval_dataset = c2_val_ts_bert,
    tokenizer = c2_tokenizer_bert
)


# Trainer2 설정
c2_trainer2 = Trainer(
    model=c1_model_roberta,
    args=training_args,
    train_dataset = c2_train_ts_roberta,
    eval_dataset = c2_train_ts_roberta,
    tokenizer = c2_tokenizer_roberta
)



#### 3) 학습

In [ ]:
c2_trainer1.train()

In [ ]:
c2_trainer2.train()

#### 4) 모델 검증평가

* 예측

In [ ]:
raw_pred1 = trainer1.predict(val_ts1)
raw_pred2 = trainer2.predict(val_ts2)
pred1 = raw_pred1.predictions.argmax(axis=1)
pred2 = raw_pred2.predictions.argmax(axis=1)

c2_raw_pred_bert = c2_trainer1.predict(c1_val_ts_bert)
c2_raw_pred_roberta = c2_trainer2.predict(c1_val_ts_roberta)

In [ ]:
true1 = raw_pred1.label_ids
true2 = raw_pred2.label_ids



* 평가

In [ ]:
try:
    aspect_list = val_ts['aspect']
except NameError:
    print("오류: 'val_ts', 'true1', 'pred1', 'true2', 'pred2' 변수를 찾을 수 없습니다.")
    print("이전 단계의 '예측' 셀을 먼저 실행했는지 확인해주세요.")
    raise


target_names = ['부정 (0)', '긍정 (1)']

all_metrics_list = []

print("=" * 70)
print("             [모델 1] 속성별 Classification Report")
print("=" * 70)

df_eval1 = pd.DataFrame({
    'aspect': aspect_list,
    'y_true': true1,
    'y_pred': pred1
})
unique_aspects1 = sorted(df_eval1['aspect'].unique())

for aspect in unique_aspects1:
    aspect_data = df_eval1[df_eval1['aspect'] == aspect]
    if len(aspect_data) == 0: continue

    y_true_aspect = aspect_data['y_true']
    y_pred_aspect = aspect_data['y_pred']

    print(f"\n--- [모델 1] '{aspect}' 속성 Report ---")
    print(classification_report(
        y_true_aspect,
        y_pred_aspect,
        target_names=target_names,
        zero_division=0
    ))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_aspect,
        y_pred_aspect,
        average='macro',
        zero_division=0
    )

    all_metrics_list.append({
        'model': 'Model 1',
        'aspect': aspect,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    })

print("\n\n")

In [ ]:
print("=" * 70)
print("             [모델 2] 속성별 Classification Report")
print("=" * 70)

df_eval2 = pd.DataFrame({
    'aspect': aspect_list,
    'y_true': true2,
    'y_pred': pred2
})
unique_aspects2 = sorted(df_eval2['aspect'].unique())

for aspect in unique_aspects2:
    aspect_data = df_eval2[df_eval2['aspect'] == aspect]
    if len(aspect_data) == 0: continue

    y_true_aspect = aspect_data['y_true']
    y_pred_aspect = aspect_data['y_pred']

    print(f"\n--- [모델 2] '{aspect}' 속성 Report ---")
    print(classification_report(
        y_true_aspect,
        y_pred_aspect,
        target_names=target_names,
        zero_division=0
    ))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_aspect,
        y_pred_aspect,
        average='macro',
        zero_division=0
    )

    all_metrics_list.append({
        'model': 'Model 2',
        'aspect': aspect,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    })

print("\n\n")

In [ ]:
print("=" * 70)
print("         [모델 1 vs 모델 2] 속성별 성능 비교 시각화")
print("=" * 70)

df_all_metrics = pd.DataFrame(all_metrics_list)

df_all_metrics_long = df_all_metrics.melt(
    id_vars=['model', 'aspect'],
    value_vars=['precision', 'recall', 'f1_score'],
    var_name='metric',
    value_name='score'
)

g = sns.catplot(
    data=df_all_metrics_long,
    x='aspect',
    y='score',
    hue='model',
    col='metric',
    kind='bar',
    height=6,
    aspect=1.5,
    legend_out=True
)

g.fig.suptitle('[모델 1 vs 모델 2] 속성(Aspect)별 Macro 점수 비교', y=1.03, fontsize=18)
g.set_axis_labels('속성 (Aspect)', '점수 (Score)')
g.set(ylim=(0, 1.05))
g.set_xticklabels(rotation=45, ha='right')
g.set_titles("{col_name}", size=14)

plt.tight_layout()
plt.show() # 그래프 출력